#### The Scenario
- Warehouses: A (Capacity 100), B (Capacity 120).
- Demand: X (80), Y (60), Z (70).
- Costs: Standard variable shipping costs.
- The Twist (Fixed Cost): Opening the route Warehouse A $\to$ Cafe Z costs a flat $50 fee plus the variable cost. All other routes are free to open.

In [8]:
import pulp

# ==========================================
# DATA SETUP
# ==========================================
warehouses = ['A', 'B']
cafes = ['X', 'Y', 'Z']

supply = {'A': 100, 'B': 125}
demand = {'X': 80, 'Y': 60, 'Z': 70}

# Variable Costs (Cost per unit)
var_costs = {
    'A': {'X': 2, 'Y': 4, 'Z': 5},
    'B': {'X': 3, 'Y': 1, 'Z': 6}
}

# Fixed Costs (One-time fee to open a route)
# We only have one fixed cost for A->Z, others are 0
fixed_costs = {
    'A': {'X': 0, 'Y': 0, 'Z': 70}, 
    'B': {'X': 0, 'Y': 0, 'Z': 0}
}


In [ ]:

# We store this for our manual calculation later
FIXED_FEE_A_Z = 70 

# ==========================================
# 2. THE MODEL (Basic LP)
# ==========================================
model = pulp.LpProblem("Scenario_A_Pay_Fixed_Cost", pulp.LpMinimize)

# Decision Variables (Continuous)
x = pulp.LpVariable.dicts("Route", (warehouses, cafes), lowBound=0, cat='Continuous')

# Objective: Minimize Variable Costs ONLY
# (The solver doesn't know about the $70 fee yet)
model += pulp.lpSum([x[w][c] * costs[w][c] for w in warehouses for c in cafes])

# Constraints
for w in warehouses:
    model += pulp.lpSum([x[w][c] for c in cafes]) <= supply[w], f"Supply_{w}"

for c in cafes:
    model += pulp.lpSum([x[w][c] for w in warehouses]) >= demand[c], f"Demand_{c}"

# ==========================================
# 3. SOLVE
# ==========================================
model.solve(pulp.PULP_CBC_CMD(msg=False))

# ==========================================
# 4. MANUAL COST ADJUSTMENT
# ==========================================
# Get the raw variable cost from the solver
variable_cost = pulp.value(model.objective)

# Check if Route A->Z was used
flow_A_Z = x['A']['Z'].varValue
is_route_used = flow_A_Z > 0

# Calculate Total Landed Cost
total_fixed_cost = FIXED_FEE_A_Z if is_route_used else 0
total_landed_cost = variable_cost + total_fixed_cost

# ==========================================
# 5. FINAL REPORT
# ==========================================
print("=== SIMPLE LINEAR PROGRAMMING ===")
print(f"1. Variable Shipping Cost: ${variable_cost}")
print(f"2. Fixed Cost Applied:     ${total_fixed_cost} (Route A->Z flow: {flow_A_Z})")
print(f"--------------------------------------")
print(f"3. TOTAL LANDED COST:      ${total_landed_cost}")
print("======================================\n")

print("Detailed Schedule:")
for w in warehouses:
    for c in cafes:
        if x[w][c].varValue > 0:
            print(f"  Ship {x[w][c].varValue} from {w} -> {c}")

=== SCENARIO REPORT (With Fixed Cost Applied) ===
1. Variable Shipping Cost: $620.0
2. Fixed Cost Applied:     $70 (Route A->Z flow: 70.0)
--------------------------------------
3. TOTAL LANDED COST:      $690.0

Detailed Schedule:
  Ship 30.0 from A -> X
  Ship 70.0 from A -> Z
  Ship 50.0 from B -> X
  Ship 60.0 from B -> Y


#### Important Logic Note: The "Fix and Re-solve" Technique
There is a catch in optimization math: You cannot get Sensitivity Analysis (Shadow Prices) directly from an Integer (MILP) problem.

Why: Shadow prices rely on "smooth" calculus curves. Integer problems are "stepped" (like a staircase), so the derivative is undefined.

- The Workaround: We use a two-phase approach widely used in industry:

    - Phase 1 (MILP): Solve the hard problem with Binary variables to decide which routes to open.

    - Phase 2 (LP): Lock in those route decisions. Re-run the problem as a simple Linear Program (continuous) to generate the Shadow Prices.

##### The Mathematical Trick: "Big M"
We cannot write if x > 0: cost += 50 directly in linear algebra. Instead, we use a binary "switch" variable and a technique called the Big M Constraint.
- Binary Variable ($y$): Let $y_{AZ} = 1$ if we use the route A->Z, and $0$ if we don't.
- The Objective: Minimize $Costs + (50 \times y_{AZ})$.
- The Constraint: We need to force $y_{AZ}$ to be 1 whenever $x_{AZ} > 0$.
        $$x_{AZ} \le M \times y_{AZ}$$
        * If $y_{AZ} = 0$: Then $x_{AZ} \le 0$ (Route is closed).<br>
        * If $y_{AZ} = 1$: Then $x_{AZ} \le M$ (Route is open, up to capacity $M$).<br>
        * $M$ is a number large enough to never limit the flow naturally (e.g., the total supply of Warehouse A).

#### PHASE 1: SOLVING MILP (Deciding Routes)

In [14]:
# Big M (Sufficiently large number for the constraint)
M = 1000

# The MILP Model

model = pulp.LpProblem("Supply_Chain_MILP", pulp.LpMinimize)

# 1. Decision Variables
# Continuous: How much to ship
x = pulp.LpVariable.dicts("Ship", (warehouses, cafes), lowBound=0, cat='Continuous')

# Binary: Do we open the route? (1 = Yes, 0 = No)
y = pulp.LpVariable.dicts("Open", (warehouses, cafes), cat='Binary')

# 2. Objective Function
# Sum of (Variable Cost * Amount) + (Fixed Cost * BinarySwitch)
model += pulp.lpSum([
    (x[w][c] * var_costs[w][c]) + (y[w][c] * fixed_costs[w][c])
    for w in warehouses for c in cafes
])

# 3. Constraints

# Supply & Demand
for w in warehouses:
    model += pulp.lpSum([x[w][c] for c in cafes]) <= supply[w], f"Supply_{w}"

for c in cafes:
    model += pulp.lpSum([x[w][c] for w in warehouses]) >= demand[c], f"Demand_{c}"

# Logical Linking Constraints (Big M)
# If y=0 (route closed), x must be 0. If y=1, x can be anything up to M.
for w in warehouses:
    for c in cafes:
        model += x[w][c] <= M * y[w][c], f"Link_{w}_{c}"

In [11]:
# 4. Solve Phase 1
model.solve(pulp.PULP_CBC_CMD(msg=False))

print("\n=== PHASE 1 REPORT (MILP Results) ===")

print("1. Decision on Fixed Routes:")
# Check if the binary variable for the expensive route is 1 or 0
if y['A']['Z'].varValue == 1:
    print("   -> Route A -> Z is OPEN (Paid $50 fixed fee)")
else:
    print("   -> Route A -> Z is CLOSED (Saved $50 fixed fee)")

print("\n2. Optimal Shipping Schedule:")
for w in warehouses:
    for c in cafes:
        # Access the amount shipped using .varValue
        amount = x[w][c].varValue
        
        # Only print if we are actually shipping something
        if amount > 0:
            print(f"   Ship {amount} units from {w} -> {c}")

print(f"\n3. Total Cost: ${pulp.value(model.objective)}")
print("=" * 40 + "\n")

status = pulp.LpStatus[model.status]
print(f"MILP Status: {status}")
print(f"Total Cost: ${pulp.value(model.objective)}\n")

# Capture the decisions made
open_routes = {}
for w in warehouses:
    for c in cafes:
        # Store whether the optimizer decided to open the route (1.0) or close it (0.0)
        open_routes[(w,c)] = y[w][c].varValue


=== PHASE 1 REPORT (MILP Results) ===
1. Decision on Fixed Routes:
   -> Route A -> Z is CLOSED (Saved $50 fixed fee)

2. Optimal Shipping Schedule:
   Ship 80.0 units from A -> X
   Ship 5.0 units from A -> Y
   Ship 55.0 units from B -> Y
   Ship 70.0 units from B -> Z

3. Total Cost: $655.0

MILP Status: Optimal
Total Cost: $655.0



#### PHASE 2: SENSITIVITY ANALYSIS (Fixing Binary & Solving LP)

In [15]:
# We create a NEW model. This time, it is a pure Linear Program (Continuous).
# We are no longer asking "Should we open the route?" 
# We are asking "Given these open routes, what is the value of capacity?"
lp_model = pulp.LpProblem("Supply_Chain_Sensitivity", pulp.LpMinimize)

# 1. Variables (Continuous only)
x_lp = pulp.LpVariable.dicts("Ship_LP", (warehouses, cafes), lowBound=0, cat='Continuous')

# 2. Objective 
# Note: We treat the Fixed Costs as sunk constants now (or ignore them for marginal analysis).
# Usually, for shadow prices, we just look at variable costs to see operational efficiency.
lp_model += pulp.lpSum([x_lp[w][c] * var_costs[w][c] for w in warehouses for c in cafes])

# 3. Constraints

# Supply Constraints (We need to capture these to get Shadow Prices!)
supply_cons = {}
supply_cons = {}
for w in warehouses:
    # Step 1: Create the constraint object
    # We define the logic: Sum of shipments <= Supply
    current_constraint = pulp.lpSum([x_lp[w][c] for c in cafes]) <= supply[w]
    
    # Step 2: Add it to the model with a name
    lp_model += current_constraint, f"Supply_{w}"
    
    # Step 3: Store the constraint object in our dictionary for later analysis
    supply_cons[w] = current_constraint

# Demand Constraints
for c in cafes:
    lp_model += pulp.lpSum([x_lp[w][c] for w in warehouses]) >= demand[c], f"Demand_{c}"
# ENFORCE THE PHASE 1 DECISIONS
# If Phase 1 said "Close Route A->Z", we force x_lp['A']['Z'] = 0
for w in warehouses:
    for c in cafes:
        if open_routes[(w,c)] < 0.5: # If binary variable was 0
            lp_model += x_lp[w][c] == 0, f"Force_Close_{w}_{c}"

In [16]:
# 4. Solve Phase 2
lp_model.solve(pulp.PULP_CBC_CMD(msg=False))

# RESULTS & REPORTING

print("\n=== FINAL REPORT ===")

print("\n1. Optimal Shipping Plan:")
for w in warehouses:
    for c in cafes:
        amount = x_lp[w][c].varValue
        if amount > 0:
            print(f"   Ship {amount} units from {w} -> {c}")

print("\n2. Sensitivity Insights (Shadow Prices):")
for w in warehouses:
    shadow_price = supply_cons[w].pi
    slack = supply_cons[w].slack
    
    print(f"   Warehouse {w}:")
    print(f"     Status: {'Full' if slack == 0 else 'Has Space'}")
    print(f"     Shadow Price: ${shadow_price:.2f}")
    
    if shadow_price < 0:
        print(f"     -> ACTION: Expanding {w} saves ${abs(shadow_price)} per extra unit capacity.")
    elif slack > 0:
        print(f"     -> ACTION: {w} has {slack} unused space. Do not expand.")
    else:
        print(f"     -> ACTION: {w} is full, but expanding it doesn't help the current route configuration.")


=== FINAL REPORT ===

1. Optimal Shipping Plan:
   Ship 80.0 units from A -> X
   Ship 5.0 units from A -> Y
   Ship 55.0 units from B -> Y
   Ship 70.0 units from B -> Z

2. Sensitivity Insights (Shadow Prices):
   Warehouse A:
     Status: Has Space
     Shadow Price: $0.00
     -> ACTION: A has 15.0 unused space. Do not expand.
   Warehouse B:
     Status: Full
     Shadow Price: $-3.00
     -> ACTION: Expanding B saves $3.0 per extra unit capacity.


#### Sensitivity Analysis Insights
**Warehouse A:**
- Has a Shadow Price of $0.00.

    Why? Because we closed the route A->Z! Warehouse A might have leftover capacity now because it isn't serving Z.<br> Expanding it further is useless because it has no open routes to send the extra goods to (or it's already serving X and Y fully).

**Warehouse B:**
- Why exactly $3.00?<br>

In optimization, numbers are never random. Let's trace why the Shadow Price is exactly $-3.00.
- The Constraint: Warehouse B is full (Status: Full). It wants to ship more but can't.
- The "Cheap" Route: Notice that shipping from B $\to$ Y costs $1.<br>
    - Shipping from A $\to$ Y costs $4.
- The Bottleneck: Because B is full (likely filling Z's demand since A $\to$ Z is closed), it doesn't have enough leftover capacity to fully serve Y.
- The Substitution: As a result, we are forced to send some shipments from A $\to$ Y at the expensive price of $4.

If we had +1 capacity at Warehouse B:
- We could ship 1 more bag from B $\to$ Y (Cost: $1).
- We would ship 1 less bag from A $\to$ Y (Saving: $4).
- Net Savings: $\$4 - \$1 = \$3$.

The math is telling us: _"I am forced to use the expensive Warehouse A for Cafe Y because Warehouse B is full. Fix B, and I can switch to the cheaper route."_